# NEWS → VIEWS: Pipeline Runner

Run the full pipeline from Colab. Clones the repo, installs deps, sets API keys, and runs.

**What you need:**
- At least one search API key: Brave (free 2k/mo), Tavily (free 1k/mo), or Exa
- OpenRouter API key (for LLM normalization)
- Optional: YouTube Data API key, Google Sheets service account

## 1. Setup

In [ ]:
# Clone the repo and install dependencies
!git clone https://github.com/jj55222/NEWS--VIEWS.git /content/NEWS--VIEWS 2>/dev/null || (cd /content/NEWS--VIEWS && git pull)
%cd /content/NEWS--VIEWS
!pip install -q -r requirements.txt

## 2. API Keys

Enter your keys below. You need **at least one search provider** + **OpenRouter**.

Use Colab Secrets (key icon in sidebar) for persistent storage, or paste directly.

In [ ]:
import os

# Try loading from Colab Secrets first, fall back to manual entry
try:
    from google.colab import userdata
    _get = lambda key, default="": userdata.get(key) if key in userdata._storage else default
except Exception:
    _get = lambda key, default="": default

# ---- SEARCH PROVIDERS (at least one required) ----
os.environ['BRAVE_API_KEY']      = _get('BRAVE_API_KEY')      or ''   # <-- paste key or use Secrets
os.environ['TAVILY_API_KEY']     = _get('TAVILY_API_KEY')     or ''   # <-- paste key or use Secrets
os.environ['EXA_API_KEY']        = _get('EXA_API_KEY')        or ''   # <-- paste key or use Secrets

# ---- LLM (required) ----
os.environ['OPENROUTER_API_KEY'] = _get('OPENROUTER_API_KEY') or ''   # <-- paste key or use Secrets
os.environ['OPENROUTER_MODEL']   = 'deepseek/deepseek-v3.2'

# ---- YOUTUBE (optional — enables direct video evidence search) ----
os.environ['YOUTUBE_API_KEY']    = _get('YOUTUBE_API_KEY')    or ''   # <-- paste key or use Secrets

# ---- GOOGLE SHEETS (optional) ----
os.environ['SHEET_ID']              = _get('SHEET_ID')              or ''
os.environ['SERVICE_ACCOUNT_PATH']  = '/content/service_account.json'

# ---- SEARCH DEFAULTS ----
os.environ['DEFAULT_START_DATE']     = '2018-01-01'
os.environ['DEFAULT_END_DATE']       = '2025-12-31'
os.environ['MAX_RESULTS_PER_REGION'] = '30'
os.environ['MIN_ARTICLE_LENGTH']     = '500'
os.environ['MIN_PRESCORE']           = '20'

# ---- OUTPUT ----
os.environ['OUTPUT_DIR'] = '/content/NEWS--VIEWS/output'
os.environ['LOG_DIR']    = '/content/NEWS--VIEWS/output/logs'

# Validate
search_ok = any(os.environ.get(k) for k in ['BRAVE_API_KEY', 'TAVILY_API_KEY', 'EXA_API_KEY'])
llm_ok = bool(os.environ.get('OPENROUTER_API_KEY'))
yt_ok = bool(os.environ.get('YOUTUBE_API_KEY'))

print('--- Config Check ---')
print(f'Search provider: {"OK" if search_ok else "MISSING - set at least one search key above"}')
print(f'OpenRouter LLM:  {"OK" if llm_ok else "MISSING - required for normalize stage"}')
print(f'YouTube API:     {"OK" if yt_ok else "skipped (optional)"}')

if search_ok and llm_ok:
    print('\nReady to run!')
else:
    print('\n** Fill in the missing keys above before running **')

## 3. Upload Service Account (optional)

Only needed if you want to read/write regions from Google Sheets.

In [ ]:
# Uncomment to upload service_account.json for Google Sheets access
# from google.colab import files
# uploaded = files.upload()  # Select your service_account.json
# import shutil
# for fn in uploaded:
#     shutil.move(fn, '/content/service_account.json')
# print('Service account uploaded')

## 4. Dry Run (no API calls)

Test that everything connects. Uses synthetic data, costs nothing.

In [ ]:
!cd /content/NEWS--VIEWS && python run_pipeline.py --dry-run

## 5. Full Pipeline Run

Set your regions and candidate limit below. Each candidate costs ~1 LLM call + 5-9 search calls.

In [ ]:
#@title Pipeline Config { display-mode: "form" }
#@markdown **Region to search** (use Region_ID from jurisdiction_portals.py)
REGION = 'SF' #@param ['SF', 'SDP', 'VJ', 'OC', 'LC', 'BC', 'MD', 'OCS', 'JS', 'PPD', 'MPD', 'MCS', 'SPD', 'KCS', 'APD', 'CSPD', 'DPD', 'ATXPD', 'HPD', 'DPDT'] {allow-input: true}
#@markdown **Max candidates** (limits LLM + enrichment costs)
LIMIT = 5 #@param {type: "integer"}

print(f'Region: {REGION}')
print(f'Limit:  {LIMIT} candidates')
print(f'Estimated cost: ~${LIMIT * 0.01:.2f} (search + LLM)')

In [ ]:
!cd /content/NEWS--VIEWS && python run_pipeline.py --region {REGION} --limit {LIMIT}

## 6. Multi-Region Run

Run across multiple regions at once. Define your own region list.

In [ ]:
import sys
sys.path.insert(0, '/content/NEWS--VIEWS')

from src.common.config import Config
from run_pipeline import run_full

config = Config.from_env()

regions = [
    {"Region_ID": "SF",  "Metro_Tokens": "San Francisco"},
    {"Region_ID": "PPD", "Metro_Tokens": "Phoenix"},
    {"Region_ID": "BC",  "Metro_Tokens": "Fort Lauderdale | Broward County"},
    # Add more regions here...
]

LIMIT = 3  # per-run candidate limit

print(f'Running {len(regions)} regions, limit {LIMIT} candidates')
run_full(config, regions, limit=LIMIT)

## 7. View Results

In [ ]:
import json
from pathlib import Path

packet_dir = Path('/content/NEWS--VIEWS/output/packets')
packets = sorted(packet_dir.glob('*.json'), key=lambda p: p.stat().st_mtime, reverse=True)

print(f'Total packets: {len(packets)}\n')

for pf in packets:
    with open(pf) as f:
        p = json.load(f)
    rec = p.get('recommendation', '?')
    comp = p.get('composite_score', 0)
    title = p.get('incident', {}).get('source_title', 'Untitled')[:80]
    n_artifacts = len(p.get('incident', {}).get('supporting_artifacts', []))
    risk = p.get('scoring', {}).get('risk_flags', [])
    print(f'[{rec:8s}] (score={comp:3.0f}) {title}')
    print(f'           artifacts={n_artifacts}  risk={risk}')
    print()

In [ ]:
# View a specific packet in detail
if packets:
    latest = packets[0]
    with open(latest) as f:
        p = json.load(f)

    print(f"=== {latest.name} ===")
    print(f"\nTitle:          {p.get('incident', {}).get('source_title', '')}")
    print(f"URL:            {p.get('incident', {}).get('source_url', '')}")
    print(f"Recommendation: {p.get('recommendation')}")
    print(f"Composite:      {p.get('composite_score', 0):.0f}")
    print(f"Story Value:    {p.get('scoring', {}).get('story_value_score', 0):.0f}")
    print(f"Researchability:{p.get('scoring', {}).get('researchability_score', 0):.0f}")
    print(f"Risk Flags:     {p.get('scoring', {}).get('risk_flags', [])}")
    print(f"Decision:       {p.get('decision_reason', '')}")

    print(f"\n--- People ---")
    for person in p.get('incident', {}).get('people', []):
        print(f"  {person.get('name', '?'):30s}  ({person.get('role', '?')})")

    print(f"\n--- Charges ---")
    for charge in p.get('incident', {}).get('allegations_or_charges', []):
        print(f"  - {charge}")

    print(f"\n--- Evidence ({len(p.get('incident', {}).get('supporting_artifacts', []))}) ---")
    for ev in p.get('incident', {}).get('supporting_artifacts', []):
        tier = ev.get('source_tier', '?')
        etype = ev.get('evidence_type', '?')
        print(f"  [{tier:8s}] {etype:15s}  {ev.get('url', '')}")
        print(f"             {ev.get('title', '')}")

    missing = p.get('incident', {}).get('missing_evidence', [])
    if missing:
        print(f"\n--- Missing Evidence ---")
        for m in missing:
            print(f"  - {m}")
else:
    print('No packets found. Run the pipeline first.')

## 8. Download Results

In [ ]:
# Zip and download all output
import shutil
output_path = '/content/NEWS--VIEWS/output'
if Path(output_path).exists() and any(Path(output_path).rglob('*.json')):
    shutil.make_archive('/content/news_views_output', 'zip', output_path)
    from google.colab import files
    files.download('/content/news_views_output.zip')
else:
    print('No output to download. Run the pipeline first.')

## 9. Prescore Testing

Test the prescore keyword matching against any article text to see if it would pass the gate.

In [ ]:
import sys
sys.path.insert(0, '/content/NEWS--VIEWS')
from src.ingest.prescore import compute_prescore, ARTIFACT_KEYWORDS, VIDEO_PLATFORMS, LIFECYCLE_KEYWORDS, CRIME_SEVERITY_KEYWORDS

print(f'Keyword bank sizes:')
print(f'  Artifact:  {len(ARTIFACT_KEYWORDS)}')
print(f'  Platforms: {len(VIDEO_PLATFORMS)}')
print(f'  Lifecycle: {len(LIFECYCLE_KEYWORDS)}')
print(f'  Severity:  {len(CRIME_SEVERITY_KEYWORDS)}')
print(f'  TOTAL:     {len(ARTIFACT_KEYWORDS) + len(VIDEO_PLATFORMS) + len(LIFECYCLE_KEYWORDS) + len(CRIME_SEVERITY_KEYWORDS)}')
print()

# Paste any article text here to test
test_text = """
A man was sentenced to 25 years in prison for the murder of his neighbor.
Body camera footage from the police department showed officers arriving at the scene.
The interrogation video shows the suspect confessing.
"""

result = compute_prescore(test_text, region_id='SF')
print(f'SCORE: {result["score"]}  (threshold: 20)')
print(f'PASS:  {"YES" if result["score"] >= 20 else "NO"}')
print(f'\nBreakdown:')
for k, v in result['breakdown'].items():
    print(f'  {k:20s} {v:3d}')
print(f'\nMatched keywords ({len(result["matches"])}):')
for m in result['matches']:
    print(f'  - {m}')